# 对话线程

许多 LLM 应用程序具有类似聊天机器人的界面，用户和 LLM 应用程序进行多轮对话。为了跟踪这些对话，您可以使用 LangSmith 中的线程功能。

这与我们的 RAG 应用程序相关，该应用程序应该维护与用户先前对话的上下文。

### 设置

In [ ]:
# 您可以在代码中直接设置
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"  # 如果您不设置此项，跟踪将进入默认项目

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 将跟踪分组到线程中

线程是表示单个对话的跟踪序列。每个响应表示为其自己的跟踪，但这些跟踪通过成为同一线程的一部分而链接在一起。

要将跟踪关联在一起，您需要传入一个特殊的元数据键，其值是该线程的唯一标识符。

键值是该对话的唯一标识符。键名应该是以下之一：

- session_id
- thread_id
- conversation_id

值应该是一个 UUID。

In [ ]:
import uuid
thread_id = uuid.uuid4()

In [ ]:
from langsmith import traceable
from openai import OpenAI
from typing import List
import nest_asyncio
from utils import get_vector_db_retriever

openai_client = OpenAI()
nest_asyncio.apply()
retriever = get_vector_db_retriever()

@traceable(run_type="chain")
def retrieve_documents(question: str):
    return retriever.invoke(question)

@traceable(run_type="chain")
def generate_response(question: str, documents):
    formatted_docs = "\n\n".join(doc.page_content for doc in documents)
    rag_system_prompt = """您是一个问答任务的助手。
    使用以下检索到的上下文片段来回答对话中的最新问题。
    如果您不知道答案，请直接说您不知道。
    最多使用三句话，保持答案简洁。
    """
    messages = [
        {
            "role": "system",
            "content": rag_system_prompt
        },
        {
            "role": "user",
            "content": f"上下文: {formatted_docs} \n\n 问题: {question}"
        }
    ]
    return call_openai(messages)

@traceable(run_type="llm")
def call_openai(
    messages: List[dict], model: str = "gpt-4o-mini", temperature: float = 0.0
) -> str:
    return openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )

@traceable(run_type="chain")
def langsmith_rag(question: str):
    documents = retrieve_documents(question)
    response = generate_response(question, documents)
    return response.choices[0].message.content

### 现在让我们使用此 thread_id 运行我们的应用程序两次

In [ ]:
question = "如何向跟踪添加元数据？"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"thread_id": thread_id}})
print(ai_answer)

In [ ]:
question = "如何向跟踪添加标签？"
ai_answer = langsmith_rag(question, langsmith_extra={"metadata": {"thread_id": thread_id}})
print(ai_answer)

### 让我们在 LangSmith 中查看一下！